# 1. Carga del dataset --------------------------------------------


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Aqui cambien el dataset por alguno de Pokémons, jugadores de la NBA, marcas de tenis, etc.
df = pd.read_csv('names.csv')

names = df['Name'].tolist()
names

['JOSE',
 'JUAN',
 'LUIS',
 'MANUEL',
 'ANTONIO',
 'JESUS',
 'CARLOS',
 'FRANCISCO',
 'ALBERTO',
 'JORGE',
 'MIGUEL',
 'ANGEL',
 'JAVIER',
 'ALEJANDRO',
 'ENRIQUE',
 'VICTOR',
 'ARTURO',
 'CESAR',
 'FERNANDO',
 'PEDRO',
 'MARTIN',
 'ROBERTO',
 'EDUARDO',
 'MARIO',
 'ARMANDO',
 'SERGIO',
 'RAUL',
 'ALFREDO',
 'RAFAEL',
 'RICARDO',
 'HECTOR',
 'OSCAR',
 'GERARDO',
 'DAVID',
 'DANIEL',
 'HUGO',
 'JAIME',
 'JULIO',
 'RUBEN',
 'RAMON',
 'MARCO',
 'GABRIEL',
 'EDGAR',
 'GUADALUPE',
 'ALFONSO',
 'GUILLERMO',
 'SALVADOR',
 'OMAR',
 'IVAN',
 'HUMBERTO',
 'FELIPE',
 'ERNESTO',
 'PABLO',
 'IGNACIO',
 'GUSTAVO',
 'ANDRES',
 'ADRIAN',
 'JOEL',
 'AGUSTIN',
 'RODOLFO',
 'GILBERTO',
 'ROGELIO',
 'RENE',
 'TOMAS',
 'SAUL',
 'ISRAEL',
 'OCTAVIO',
 'VICENTE',
 'NOE',
 'GREGORIO',
 'ISMAEL',
 'NICOLAS',
 'BENJAMIN',
 'MOISES',
 'SANTIAGO',
 'EFRAIN',
 'ALONSO',
 'ABEL',
 'JOSE DE JESUS',
 'ALVARO',
 'FELIX',
 'MARCOS',
 'ADOLFO',
 'RODRIGO',
 'RAMIRO',
 'SAMUEL',
 'JOAQUIN',
 'ABRAHAM',
 'ESTEBAN',
 'ULIS

# 2. Construcción de vocabulario ----------------------------------


In [2]:

# Añadimos tokens especiales: '\t' como start-of-sequence y '\n' como end-of-sequence
all_chars = sorted(set(''.join(names)))
vocab = ['\t', '\n'] + all_chars
char2idx = {ch: i for i, ch in enumerate(vocab)}
idx2char = {i: ch for ch, i in char2idx.items()}

# Longitud máxima de secuencia (incluye tokens especiales)
maxlen = max(len(n) for n in names) + 2  # +1 para '\t', +1 para '\n'

# 3. Vectorización de los datos -----------------------------------


In [3]:
# X: input sequences; y: target sequences desplazadas un paso
X = np.zeros((len(names), maxlen), dtype='int32')
y = np.zeros((len(names), maxlen), dtype='int32')

for i, name in enumerate(names):
    seq = ['\t'] + list(name) + ['\n']
    for t, ch in enumerate(seq):
        X[i, t] = char2idx[ch]
        if t > 0:
            y[i, t-1] = char2idx[ch]

# 4. Definición del Transformer básico -----------------------------


In [4]:
embed_dim = 64   # dimensión del embedding de caracteres
ff_dim    = 128  # dimensión interna del feed-forward
num_heads = 2

class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb   = layers.Embedding(input_dim=maxlen,  output_dim=embed_dim)
    def call(self, x):
        seq_len = tf.shape(x)[1]
        positions = tf.range(start=0, limit=seq_len, delta=1)
        pos_emb = self.pos_emb(positions)
        return self.token_emb(x) + pos_emb

In [5]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation='relu'),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training):
        # Máscara causal para que cada token solo atienda a anteriores
        seq_len = tf.shape(inputs)[1]
        causal_mask = tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
        attn_out = self.att(inputs, inputs, attention_mask=causal_mask)
        attn_out = self.dropout1(attn_out, training=training)
        out1 = self.layernorm1(inputs + attn_out)
        ffn_out = self.ffn(out1)
        ffn_out = self.dropout2(ffn_out, training=training)
        return self.layernorm2(out1 + ffn_out)

In [6]:
# Compilacion del modelo
vocab_size = len(vocab)
inputs  = layers.Input(shape=(None,), dtype="int32")
x       = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)(inputs)
x       = TransformerBlock(embed_dim, num_heads, ff_dim)(x)
outputs = layers.Dense(vocab_size, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, None)]            0         
                                                                 
 token_and_position_embeddi  (None, None, 64)          2944      
 ng (TokenAndPositionEmbedd                                      
 ing)                                                            
                                                                 
 transformer_block (Transfo  (None, None, 64)          50048     
 rmerBlock)                                                      
                                                                 
 dense_2 (Dense)             (None, None, 29)          1885      
                                                                 
Total params: 54877 (214.36 KB)
Trainable params: 54877 (214.36 KB)
Non-trainable params: 0 (0.00 Byte)
_______________________

# 5. Entrenamiento -------------------------------------------------


In [7]:
history = model.fit(
    X, y,
    batch_size=32,
    epochs=20,
    validation_split=0.1
)

Epoch 1/20
8/8 [==============================] - 2s 45ms/step - loss: 2.3057 - val_loss: 1.5633
Epoch 2/20
8/8 [==============================] - 0s 10ms/step - loss: 1.3547 - val_loss: 1.2452
Epoch 3/20
8/8 [==============================] - 0s 10ms/step - loss: 1.1878 - val_loss: 1.1613
Epoch 4/20
8/8 [==============================] - 0s 9ms/step - loss: 1.0866 - val_loss: 1.1497
Epoch 5/20
8/8 [==============================] - 0s 9ms/step - loss: 1.0569 - val_loss: 1.1307
Epoch 6/20
8/8 [==============================] - 0s 9ms/step - loss: 1.0284 - val_loss: 1.0887
Epoch 7/20
8/8 [==============================] - 0s 10ms/step - loss: 1.0169 - val_loss: 1.0948
Epoch 8/20
8/8 [==============================] - 0s 9ms/step - loss: 0.9883 - val_loss: 1.0765
Epoch 9/20
8/8 [==============================] - 0s 9ms/step - loss: 0.9757 - val_loss: 1.0678
Epoch 10/20
8/8 [==============================] - 0s 9ms/step - loss: 0.9627 - val_loss: 1.0576
Epoch 11/20
8/8 [==================

# 6. Generación de nuevos nombres ---------------------------------


In [8]:
def sample_with_temperature(preds, temperature=1.0):

    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds + 1e-8) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)

    # muestrear índice según la distribución resultante
    return np.random.choice(len(preds), p=preds)

def generate_name(max_length=20, temperature=1.0):

    idxs = [char2idx['\t']]  # token de inicio
    for _ in range(max_length):
        padded = tf.expand_dims(idxs + [0]*(maxlen-len(idxs)), 0)
        
        # preds: vector de probabilidades over vocab
        preds = model(padded)[0, len(idxs)-1].numpy()

        # muestreo con temperatura
        next_i = sample_with_temperature(preds, temperature)
        if idx2char[next_i] == '\n':
            break
        idxs.append(int(next_i))
        
    # decodifica la secuencia en string
    return ''.join(idx2char[i] for i in idxs[1:])

In [10]:
for temp in [0.5, 1.0, 1.5]:
    print(f"Temperature={temp}:")
    for _ in range(3):
        print(" ", generate_name(temperature=temp))
    print()

Temperature=0.5:
  ARIS
  ARIS
  ALINAN

Temperature=1.0:
  ADRDO
  LACUNTO
  DPAEL

Temperature=1.5:
  RMAN
  ENIO
  EDEFOS

